# Application of Radial Equilibrium Equation for a Rotor - Direct Problem
## Adaptation of the code RE-ANAL by Lewis (Turbomachines Performance Analysis)

This notebook implements the **direct analysis (RE-ANAL) problem** for radial equilibrium in axial turbomachinery, as described in **Lewis Chapter 5 (Section 5.3.2)**. 
 
Unlike the *inverse (design)* problem (`RE-DES`), where we specify a desired swirl velocity $c_\theta(r)$ and calculate the required blade geometry, the *direct (analysis)* problem starts with a **fixed blade geometry** (defined by the relative outlet flow angle $\beta_2(r)$) and predicts the resulting axial velocity profile $c_x(r)$, absolute tangential velocity $c_\theta(r)$, and overall fan performance.

The necessary modules are imported

In [10]:
import numpy as np
import pandas as pd
pd.set_option('display.precision', 2)
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid, trapezoid

Let's input the general dimensions and target parameters of the fan. These are consistent with the **TXBR-250 ECOWATT** prototype fan studied in previous sessions.

In [11]:
rho = 1.2                           # Density of air, kg/m³
Dh = 0.064                          # Diameter of hub, m
rh = Dh/2                           # Radius of hub, m
Dt = 0.25                           # Diameter of tip, m
rt = Dt/2                           # Radius of tip, m
h = rh/rt                           # Hub-tip ratio
rrms = np.sqrt(0.5*(rh*rh+rt*rt))   # RMS radius, m
n = 10                              # number of outputs
m = 601                             # number of interpolation points
r = np.linspace(rh,rt,m)            # Discretization of the radius for interpolation, m
Qdata = 1716                        # Flow rate, m³/h
Qdata = Qdata/3600                  # Flow rate, m³/s
omega = 2275                        # Rotational speed, rpm
omega = omega*np.pi/30              # Rotational speed, rad/s
Delta_p0_target = 75                # Pressure rise, Pa

ctrms = Delta_p0_target/(rho*omega*rrms)         # c_theta,rms, m/s
cxm = Qdata/(np.pi*(rt*rt-rh*rh))   # c_x,rms, m/s
print("Flow rate = {:0.4f} m³/s".format(Qdata))
print("Hub to tip ratio = {:0.4f}".format(h))
print("rms = {:.4f} m".format(rrms))
print("ctheta_rms = {:.4f} m/s".format(ctrms))
print("cx_rms = {:.4f} m/s".format(cxm))


Flow rate = 0.4767 m³/s
Hub to tip ratio = 0.2560
rms = 0.0912 m
ctheta_rms = 2.8754 m/s
cx_rms = 10.3916 m/s


In a direct analysis problem, the blade geometry is defined by the outlet flow angle $\beta_2(r)$ at several radial stations.
 
Below, we define the relative flow angles $\beta_2$ at 10 radial stations. 

In [12]:
rdata = np.linspace(rh, rt, 10)

# Relative outlet flow angle beta2 (degrees) from a successful design run
# Note: For a stator analysis, setting omega = 0 and inputting absolute flow angles alpha2 works identically!
beta2_input = np.array([
    31.46,
    49.72,
    57.84,
    61.14,
    62.10,
    61.99,
    61.40,
    60.65,
    59.85,
    59.08
])

# Display input geometry
df_geom = pd.DataFrame({
    "Station Radius (m)": rdata,
    "beta_2 Flow Angle (deg)": beta2_input
})
df_geom

,Station Radius (m),beta_2 Flow Angle (deg)
0,0.03,31.46
1,0.04,49.72
2,0.05,57.84
3,0.06,61.14
4,0.07,62.10
5,0.08,61.99
6,0.09,61.40
7,0.10,60.65
8,0.11,59.85
9,0.12,59.08


From Lewis Section 5.3.2, the first-order linear differential equation governing the axial velocity downstream of a rotor $c_{x2}(r)$ is:

$$\frac{\text{d}c_{x2}}{\text{d}r} + f_1(r) c_{x2} = f_2(r)$$
 
Where:
$$f_1(r) = \frac{\tan \beta_2}{r (1 + \tan^2 \beta_2)} \frac{\text{d}(r \tan \beta_2)}{\text{d}r}$$
$$f_2(r) = \frac{2 \omega \tan \beta_2}{1 + \tan^2 \beta_2}$$
 
The solution is found iteratively by integrating:
$$c_{x2}(r) = L(r, c_{x2}) + K_1$$
 
Where $L(r, c_{x2}) = \int_{r_h}^r \left( -c_{x2} f_1(r) + f_2(r) \right) \text{d}r$, and the integration constant $K_1$ is updated to satisfy the target mass flow rate $Q$.


In [13]:
m = 600
r = np.linspace(rh, rt, m + 1)
dr = (rt - rh) / m

# 1. Interpolate input beta2 distribution onto the fine integration grid
beta2_spline = CubicSpline(rdata, beta2_input)
beta2 = beta2_spline(r)
tanbeta2 = np.tan(np.radians(beta2))

# 2. Pre-calculate terms for f1 and f2 evaluated at midpoints rm
rm = 0.5 * (r[:-1] + r[1:])
tanbeta2m = 0.5 * (tanbeta2[:-1] + tanbeta2[1:])

# Calculate the derivative d(r * tan(beta2)) / dr using central/forward differences on the fine grid
r_tanbeta2 = r * tanbeta2
drtanbeta2 = (r_tanbeta2[1:] - r_tanbeta2[:-1]) / dr

# Evaluate f1 and f2 at midpoints rm
f1 = (tanbeta2m / rm) * drtanbeta2 / (1 + tanbeta2m**2)
f2 = 2 * omega * tanbeta2m / (1 + tanbeta2m**2)


We define the integration operator $L(r, c_x)$

In [14]:
def compute_L(cx_arr):
    # Midpoint axial velocity for integration step
    cx_m = 0.5 * (cx_arr[:-1] + cx_arr[1:])
    integrand = -cx_m * f1 + f2
    L_arr = np.zeros(m + 1)
    L_arr[1:] = np.cumsum(integrand * dr)
    return L_arr

## The Iteration Loop

We solve the implicit relationship for $c_{x2}(r)$ and match the target flow rate $Q_{\text{data}}$ using the successive approximation scheme (Lewis Equation 5.47).

In [15]:
# Initialize axial velocity with the mean value
cx_anal = np.full(m + 1, cxm)

# Initial integration of L
L = compute_L(cx_anal)
Lrms = CubicSpline(r, L)(rrms)
k1 = cxm - Lrms  # First approximation of K1

max_iter = 100
tolerance = 1e-6

print(f"{'Iteration':^12} | {'K1':^12} | {'K2':^12} | {'Error (%)':^12}")
print("—" * 55)

for j in range(1, max_iter + 1):
    cxnew = L + k1
    cx_anal = 0.5 * (cx_anal + cxnew) # Under-relaxation for stable convergence
    L = compute_L(cx_anal)
    
    # Calculate mass flow integral (r * L) over r
    integral_rL = trapezoid(r * L, r)
    k2 = cxm - 2 * integral_rL / (rt**2 - rh**2)
    
    # Check convergence error
    error = np.abs((k1 - k2) / k1)
    k1 = 0.5 * (k1 + k2)
    
    if j % 5 == 0 or error < tolerance:
        print(f"{j:^12d} | {k1:12.6f} | {k2:12.6f} | {error * 100:12.2e}")
        
    if error < tolerance:
        print(f"\nConvergence achieved successfully in {j} iterations!")
        break

 Iteration   |      K1      |      K2      |  Error (%)  
———————————————————————————————————————————————————————
     5       |     8.655982 |     8.299486 |     7.91e+00
     10      |     7.602169 |     7.462439 |     3.61e+00
     15      |     7.189599 |     7.134743 |     1.51e+00
     20      |     7.027374 |     7.005791 |     6.12e-01
     25      |     6.963537 |     6.955044 |     2.44e-01
     30      |     6.938416 |     6.935073 |     9.63e-02
     35      |     6.928530 |     6.927215 |     3.80e-02
     40      |     6.924640 |     6.924122 |     1.49e-02
     45      |     6.923109 |     6.922905 |     5.88e-03
     50      |     6.922506 |     6.922426 |     2.32e-03
     55      |     6.922269 |     6.922238 |     9.11e-04
     60      |     6.922176 |     6.922164 |     3.59e-04
     65      |     6.922139 |     6.922135 |     1.41e-04
     67      |     6.922132 |     6.922129 |     9.72e-05

Convergence achieved successfully in 67 iterations!


Now that the velocity field $c_x(r)$ has been determined, we can calculate the local absolute tangential velocity $c_\theta(r)$ and local total pressure rise $\Delta p_0(r)$ across the blade span:

$$c_{\theta}(r) = \omega r - c_x(r) \tan \beta_2(r)$$
$$\Delta p_0(r) = \rho \omega r c_\theta(r)$$

In [16]:
# Calculate local tangential velocity and pressure rise on the fine grid
ctheta_anal = omega * r - cx_anal * tanbeta2
Delta_p0 = rho * omega * r * ctheta_anal

# Integrate to find the overall calculated flow rate and area-averaged pressure rise
Q_calc = 2 * np.pi * trapezoid(r * cx_anal, r)
Delta_p0_avg = 2 * trapezoid(r * Delta_p0, r) / (rt**2 - rh**2)

print(f"\nOverall Fan Performance Summary:")
print(f"Calculated Flow Rate Q     = {Q_calc * 3600:.1f} m³/h (Target: {Qdata*3600:.1f} m³/h)")
print(f"Flow Rate Error            = {np.abs(Qdata - Q_calc)/Qdata * 100:.2e}%")
print(f"Area-weighted Avg Pressure = {Delta_p0_avg:.2f} Pa (Target: {Delta_p0_target:.1f} Pa)")
print(f"Pressure Rise Error        = {np.abs(Delta_p0_target - Delta_p0_avg)/Delta_p0_target * 100:.2f}%")


Overall Fan Performance Summary:
Calculated Flow Rate Q     = 1716.0 m³/h (Target: 1716.0 m³/h)
Flow Rate Error            = 4.91e-05%
Area-weighted Avg Pressure = 76.69 Pa (Target: 75.0 Pa)
Pressure Rise Error        = 2.26%


## Results Table (Pandas DataFrame)
Let's extract the radial solutions at the 10 output stations to display them in a clean table.

In [17]:
cx_stations = CubicSpline(r, cx_anal)(rdata)
ct_stations = CubicSpline(r, ctheta_anal)(rdata)
dp0_stations = CubicSpline(r, Delta_p0)(rdata)
beta1_stations = np.rad2deg(np.arctan(omega * rdata / cx_stations))

results_list = []
for i in range(rdata.size):
    results_list.append({
        "Radius (m)": rdata[i],
        "c_x (m/s)": cx_stations[i],
        "c_theta (m/s)": ct_stations[i],
        "beta_1 (deg)": beta1_stations[i],
        "beta_2 (deg)": beta2_input[i],
        "Local Delta p0 (Pa)": dp0_stations[i]
    })

# Convert to DataFrame and round values
df_results = pd.DataFrame(results_list)

df_results

,Radius (m),c_x (m/s),c_theta (m/s),beta_1 (deg),beta_2 (deg),Local Delta p0 (Pa)
0,0.03,6.92,3.39,47.76,31.46,31.00
1,0.04,6.63,2.27,56.69,49.72,27.42
2,0.05,6.71,1.87,61.86,57.84,28.21
3,0.06,7.25,1.85,64.22,61.14,33.40
4,0.07,8.16,2.05,64.95,62.10,43.01
5,0.08,9.34,2.38,64.90,61.99,56.96
6,0.09,10.68,2.80,64.50,61.40,75.36
7,0.10,12.13,3.29,63.99,60.65,98.09
8,0.11,13.65,3.82,63.45,59.85,125.30
9,0.12,15.21,4.39,62.95,59.08,156.85


In [18]:
df_results['c_theta (m/s)'].to_numpy()

array([3.38834952, 2.26576301, 1.87339051, 1.85468094, 2.05138956,
       2.38135795, 2.80430445, 3.28864899, 3.82218414, 4.38921487])